# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sujithauday/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Among pages that already get meaningful traffic and already show CTR below what their position tier normally earns, predict which ones will keep declining — those go to the top of the SEO team’s queue.
This is a CTR/Engagement Opportunity Scoring lane.
The ML task is ranking/scoring, producing an opportunity score and a top‑50 priority list for SEO.

One row = one web page(content_id)

In [34]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute(f"""
    CREATE SECRET hf_token_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

rel = "hf://datasets/FlyRank/internship-warehouse"
table = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
con.sql(f"""
    SELECT COUNT(*)
    FROM {table}
""").show()
# Column names

schema = con.sql(f"DESCRIBE SELECT * FROM {table}").df()
print(schema["column_name"].tolist())
top3 = con.sql(f"SELECT * FROM {table} LIMIT 3").df()
print(top3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0           

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# sum(gsc_impressions), sum(gsc_clicks), sum(ga4_pageview), sum(ga4_sessions), sum(ga4_engaged_sessions),

FACT_DAILY = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {FACT_DAILY}
""").df()
# print(summary.columns.tolist())
print(summary)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  clients  content_items first_report_date last_report_date
0    78835655       70         427292        2025-01-27       2026-06-30


In [ ]:
MONTH = '2025-02'
table2 = f"""
    read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
"""
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM {table2}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate raw-grain rows found in month={MONTH}: {len(grain_check)}')
if len(grain_check) == 0:
    print('Result: zero rows — the grain holds (report_date × client_hash_id × content_hash_id).')
else:
    display(grain_check)

Duplicate raw-grain rows found in month=2025-02: 0
Result: zero rows — the grain holds (report_date × client_hash_id × content_hash_id).


In [19]:
# get data for 4 months:
MONTHS = ['2026-02', '2026-03', '2026-04', '2026-05']
paths = ", ".join([f"'{rel}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS])

coverage = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) AS first_seen,
        MAX(report_date) AS last_seen,
        COUNT(DISTINCT report_date) AS days_present
    FROM read_parquet([{paths}])
    GROUP BY client_hash_id
    ORDER BY first_seen
""").df()

print(f"Clients with any data in Feb - May 2026: {len(coverage)} of 70 total")
print(coverage.sort_values(by = "days_present", ascending = True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with any data in Feb - May 2026: 70 of 70 total
             client_hash_id first_seen  last_seen  days_present
69  client_aef6ffea193da149 2026-05-28 2026-05-31             4
67  client_a22068e339bf95f5 2026-05-22 2026-05-31            10
68  client_04660893ae39614a 2026-05-22 2026-05-31            10
64  client_c353557474475e51 2026-04-30 2026-05-31            22
8   client_861cdcccf8049915 2026-02-01 2026-02-23            23
..                      ...        ...        ...           ...
38  client_def0955f7a377868 2026-02-01 2026-05-31           120
37  client_a60a11451483af1c 2026-02-01 2026-05-31           120
36  client_795153d5b7850ccf 2026-02-01 2026-05-31           120
35  client_a1203ffecad62470 2026-02-01 2026-05-31           120
34  client_59256b0571e0c970 2026-02-01 2026-05-31           120

[70 rows x 4 columns]


In [16]:
# Grain
rel = "hf://datasets/FlyRank/internship-warehouse"
MONTHS = ['2026-05', '2026-03', '2026-04', '2026-02']
paths = ", ".join([f"'{rel}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS])

coverage = con.sql(f"""
    SELECT
        DISTINCT month, count(DISTINCT report_date)
    FROM read_parquet([{paths}])
    WHERE client_hash_id IN (
    SELECT
  client_hash_id
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
WHERE is_active = TRUE AND  has_gsc_access = TRUE  AND has_ga4_access = TRUE
AND gsc_data_start IS NOT NULL AND ga4_data_start IS NOT NULL)
GROUP BY month
""").df()

#print(f"Clients with any data in Jan - Apr 2026: {len(coverage)} of 70 total")
#print(coverage.sort_values(by = "days_present", ascending = True).head(30))
print(coverage)
#print(f"Clients with full 120-day coverage (Feb 1 – May 30, 2026): {exact_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     month  count(DISTINCT report_date)
0  2026-03                           31
1  2026-02                           28
2  2026-05                           31
3  2026-04                           30


In [38]:
rel = "hf://datasets/FlyRank/internship-warehouse"
MONTHS = ['2026-05', '2026-03', '2026-04', '2026-02']
paths = ", ".join([f"'{rel}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS])

con.sql(f"""
    SELECT *
    FROM read_parquet([{paths}])
    LIMIT 1
""").df().columns
x = con.sql(f"""
    SELECT client_hash_id, content_hash_id, month,
    SUM(gsc_clicks) FILTER (WHERE month = '2026-02') AS clicks_2026_prev30d,
    SUM(gsc_clicks) FILTER (WHERE month = '2026-03') AS clicks_2026_last30d,
    SUM(gsc_clicks) FILTER (WHERE month = '2026-04') AS clicks_2026_30d,
    SUM(gsc_clicks) FILTER (WHERE month = '2026-05') AS clicks_2026_f30d,

    SUM(gsc_impressions) FILTER (WHERE month = '2026-02') AS impressions_prev30d,
    SUM(gsc_impressions) FILTER (WHERE month = '2026-03') AS impressions_last30d,
    SUM(gsc_impressions) FILTER (WHERE month = '2026-04') AS impressions_30d,
    SUM(gsc_impressions) FILTER (WHERE month = '2026-05') AS impressions_f30d,

    SUM(ga4_pageviews) FILTER (WHERE month = '2026-02') AS pageviews_prev30d,
    SUM(ga4_pageviews) FILTER (WHERE month = '2026-03') AS pageviews_last30d,
    SUM(ga4_pageviews) FILTER (WHERE month = '2026-04') AS pageviews_30d,
    SUM(ga4_pageviews) FILTER (WHERE month = '2026-05') AS pageviews_f30d,

    SUM(ga4_sessions) FILTER (WHERE month = '2026-02') AS sessions_prev30d,
    SUM(ga4_sessions) FILTER (WHERE month = '2026-03') AS sessions_last30d,
    SUM(ga4_sessions) FILTER (WHERE month = '2026-04') AS sessions_30d,
    SUM(ga4_sessions) FILTER (WHERE month = '2026-02') AS sessions_f30d,

    SUM(ga4_engaged_sessions) FILTER (WHERE month = '2026-02') AS sessions_prev30d,
    SUM(ga4_engaged_sessions) FILTER (WHERE month = '2026-03') AS sessions_last30d,
    SUM(ga4_engaged_sessions) FILTER (WHERE month = '2026-04') AS sessions_30d,
    SUM(ga4_engaged_sessions) FILTER (WHERE month = '2026-02') AS sessions_f30d,

    FROM read_parquet([{paths}])
    WHERE client_hash_id IN (
    SELECT
  client_hash_id
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
WHERE is_active = TRUE AND  has_gsc_access = TRUE  AND has_ga4_access = TRUE
AND gsc_data_start IS NOT NULL AND ga4_data_start IS NOT NULL)
GROUP BY client_hash_id, content_hash_id, month


""").df()
print(x.count())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

client_hash_id         800583
content_hash_id        800583
month                  800583
clicks_2026_prev30d    174415
clicks_2026_last30d    190881
clicks_2026_30d        208953
clicks_2026_f30d       226334
impressions_prev30d    174415
impressions_last30d    190881
impressions_30d        208953
impressions_f30d       226334
pageviews_prev30d      107213
pageviews_last30d      190881
pageviews_30d          208953
pageviews_f30d         226334
sessions_prev30d       107213
sessions_last30d       190881
sessions_30d           208953
sessions_f30d          107213
sessions_prev30d_1     107213
sessions_last30d_1     190881
sessions_30d_1         208953
sessions_f30d_1        107213
dtype: int64


In [ ]:
 SUM(ga4_sessions) FILTER (WHERE month = '2026-02') AS sessions_,
    SUM(ga4_engaged_sessions) AS monthly_engaged_sessions
    ORDER BY monthly_impressions DESC

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.